In [3]:

# DAY 4 HOMEWORK - Cell 1: Environment Setup
# CPU only. No API key required.

import importlib.util
import subprocess
import sys

print("Python version:", sys.version)

# Preserve Colab's preinstalled PyTorch
if importlib.util.find_spec('torch') is None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'torch'
    ])

# Install compatible libraries
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'transformers==4.49.0',
    'sentencepiece==0.2.1'
])

import torch
import transformers
import sentencepiece

print('Torch runtime:', torch.__version__)
print('Transformers:', transformers.__version__)
print('SentencePiece:', sentencepiece.__version__)
print('Environment setup completed successfully!')


Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Torch runtime: 2.14.1+cpu
Transformers: 4.49.0
SentencePiece: 0.2.1
Environment setup completed successfully!


In [5]:
# Independent real-model evaluation homework solution. Same pinned dependencies as demo.
import json
import time
from pathlib import Path


def run_evaluation_homework():
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    torch.set_num_threads(2)
    model_id = 'google/flan-t5-small'
    revision = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab'
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, trust_remote_code=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id, revision=revision,
        trust_remote_code=False, use_safetensors=True).to('cpu').eval()
    context = 'You can check your application status using its reference number.'
    questions = ['How can I check my application status?',
                 'What do I need to check my application status?',
                 'How can I find out the status of my application?']
    rows = []
    for case_number, question in enumerate(questions, 1):
        prompt = f'Answer the question using the context.\nContext: {context}\nQuestion: {question}\nAnswer:'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=256)
        start = time.perf_counter()
        with torch.inference_mode():
            tokens = model.generate(**inputs, max_new_tokens=48, do_sample=False, num_beams=1)
        answer = tokenizer.decode(tokens[0], skip_special_tokens=True).strip()
        assert answer, 'Empty generation; inspect runtime.'
        # A deliberately weak measurable proxy, not a semantic judge.
        keyword_hit = 'reference' in answer.lower() and 'number' in answer.lower()
        rows.append({'id': f'E{case_number:02d}', 'question': question, 'answer': answer, 'reference_keyword_hit': keyword_hit,
                     'input_tokens': int(inputs['input_ids'].shape[-1]),
                     'output_tokens_including_special': int(tokens.shape[-1]),
                     'inference_seconds': round(time.perf_counter() - start, 3)})
    result = {'mode': 'real_local_model', 'model': model_id, 'model_revision': revision,
              'generation': {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 48},
              'metric': 'reference+number keyword proxy; NOT semantic accuracy',
              'keyword_hits': sum(x['reference_keyword_hit'] for x in rows), 'total': len(rows),
              'external_inference_api_calls': 0, 'cases': rows,
              'manual_review': 'Check every answer for correct meaning and unsupported facts. Record failures; do not edit expected criteria to force a pass.'}
    out = Path('murshid_live_outputs'); out.mkdir(exist_ok=True)
    (out / 'evaluation_result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(json.dumps(result, indent=2))
    return result




In [6]:
result = run_evaluation_homework()


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

{
  "mode": "real_local_model",
  "model": "google/flan-t5-small",
  "model_revision": "0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab",
  "generation": {
    "do_sample": false,
    "num_beams": 1,
    "max_new_tokens": 48
  },
  "metric": "reference+number keyword proxy; NOT semantic accuracy",
  "keyword_hits": 3,
  "total": 3,
  "external_inference_api_calls": 0,
  "cases": [
    {
      "id": "E01",
      "question": "How can I check my application status?",
      "answer": "Use its reference number",
      "reference_keyword_hit": true,
      "input_tokens": 34,
      "output_tokens_including_special": 6,
      "inference_seconds": 0.737
    },
    {
      "id": "E02",
      "question": "What do I need to check my application status?",
      "answer": "Reference number",
      "reference_keyword_hit": true,
      "input_tokens": 36,
      "output_tokens_including_special": 4,
      "inference_seconds": 0.333
    },
    {
      "id": "E03",
      "question": "How can I find out the stat

### Manual Evaluation

All three generated answers are supported by the synthetic FAQ context and correctly identify the reference number as the information needed to check an application status.

- **E01:** Correct and relevant.
- **E02:** Correct, although the answer is shorter and less explanatory.
- **E03:** Correct and relevant.

The keyword indicator returned 3/3 hits, but this metric only checks for the presence of the words "reference" and "number"; it does not measure semantic accuracy.

**Suggested Improvement:** Improve the response format by encouraging complete, user-friendly sentences, especially for short outputs such as E02.

No unsupported fees, dates, or application-status claims were observed.